# 🐟 Gemastik IUU Fishing Detection - Server Setup

Run each cell sequentially. This will:
1. Clone the repo
2. Download raw data (Zenodo + GFW)
3. Install dependencies
4. Run the full pipeline (Phase 1-7)
5. Verify output

**Team:** Toni, Nafi, Rhendy | **Competition:** Gemastik XIX 2026

In [ ]:
# Cell 1: Clone repo
import os
import subprocess

REPO_DIR = os.path.expanduser("~/gemastik")

if os.path.exists(REPO_DIR):
    print(f"Repo exists at {REPO_DIR}, pulling latest...")
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    print("Cloning repo...")
    subprocess.run([
        "git", "clone",
        "https://github.com/RedEye1605/iuu-fishing-detection.git",
        REPO_DIR
    ], check=True)

print(f"✅ Repo ready at {REPO_DIR}")

In [ ]:
# Cell 2: Download large data files from GitHub Release
# Zenodo vessel registry + monthly effort files (~700MB)
import urllib.request
import sys

REPO = "RedEye1605/iuu-fishing-detection"
TAG = "v1.0-data"
ZENODO_DIR = os.path.join(REPO_DIR, "data", "raw", "zenodo")
os.makedirs(ZENODO_DIR, exist_ok=True)

FILES = [
    "fishing-vessels-v3.csv",
    "fleet-monthly-csvs-10-v3-2020.zip",
    "fleet-monthly-csvs-10-v3-2021.zip",
    "fleet-monthly-csvs-10-v3-2022.zip",
    "fleet-monthly-csvs-10-v3-2023.zip",
    "fleet-monthly-csvs-10-v3-2024.zip",
]

for fname in FILES:
    fpath = os.path.join(ZENODO_DIR, fname)
    if os.path.exists(fpath):
        print(f"  ⏭️  {fname} exists, skipping")
        continue
    url = f"https://github.com/{REPO}/releases/download/{TAG}/{fname}"
    print(f"  ⬇️  Downloading {fname}...")
    urllib.request.urlretrieve(url, fpath)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  ✅ {fname} ({size_mb:.1f} MB)")

print(f"\n✅ All data files downloaded")
print(f"   Zenodo dir: {ZENODO_DIR}")

In [ ]:
# Cell 3: Install dependencies (includes PyTorch + PyG)
import subprocess

print("Installing core dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "pandas>=2.2", "scikit-learn>=1.4",
    "requests>=2.31", "matplotlib>=3.8", "seaborn>=0.13"
], check=True)

print("Installing PyTorch (CPU-only) + PyTorch Geometric...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch>=2.2", "torch_geometric>=2.5",
    "--index-url", "https://download.pytorch.org/whl/cpu",
    "--extra-index-url", "https://data.pyg.org/whl/torch-2.2.0+cpu.html"
], check=True)

print("✅ All dependencies installed")

# Verify
import torch
import pandas as pd
import numpy as np
import sklearn
print(f"  torch {torch.__version__}")
print(f"  pandas {pd.__version__}, numpy {np.__version__}, sklearn {sklearn.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Cell 4: Add project to Python path and verify data
import sys
sys.path.insert(0, REPO_DIR)

DATA_DIR = os.path.join(REPO_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

# Check raw data
gfw_dir = os.path.join(RAW_DIR, "gfw")
zenodo_dir = os.path.join(RAW_DIR, "zenodo")

gfw_files = [f for f in os.listdir(gfw_dir) if f.endswith('.gz')] if os.path.exists(gfw_dir) else []
zenodo_files = os.listdir(zenodo_dir) if os.path.exists(zenodo_dir) else []

print(f"GFW raw files: {len(gfw_files)}")
print(f"Zenodo files: {len(zenodo_files)}")
print(f"\nGFW files: {gfw_files}")
print(f"Zenodo files: {zenodo_files}")

if len(gfw_files) < 4:
    print("\n⚠️  Missing GFW raw data! These should be in the repo.")
else:
    print("\n✅ All raw data present")

In [ ]:
# Cell 5: Run full pipeline (Phase 1-7)
# This takes ~15-20 minutes total
import time

start = time.time()
result = subprocess.run(
    [sys.executable, os.path.join(REPO_DIR, "scripts", "run_pipeline.py")],
    cwd=REPO_DIR,
    capture_output=False,
    timeout=1800  # 30 min max
)
elapsed = time.time() - start

if result.returncode == 0:
    print(f"\n✅ Pipeline complete ({elapsed/60:.1f} minutes)")
else:
    print(f"\n❌ Pipeline failed with exit code {result.returncode}")

In [ ]:
# Cell 6: Verify output
import os

expected_files = [
    "gfw_events_labeled.parquet",
    "gfw_events_full.parquet",
    "vessel_node_features.parquet",
    "encounter_edges.parquet",
    "colocation_edges.parquet",
    "snapshot_metadata.parquet",
    "feature_scaler.pkl",
    "graph_snapshots.pkl",
]

print("=== Output Verification ===")
all_ok = True
for f in expected_files:
    fpath = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  ✅ {f} ({size:.1f} MB)")
    else:
        print(f"  ❌ {f} MISSING")
        all_ok = False

# Check model data
model_dir = os.path.join(PROCESSED_DIR, "model")
if os.path.exists(model_dir):
    model_files = os.listdir(model_dir)
    print(f"\n  Model data: {len(model_files)} files")
    for f in sorted(model_files):
        print(f"    ✅ {f}")

# Quick data check
if os.path.exists(os.path.join(PROCESSED_DIR, "vessel_node_features.parquet")):
    import pandas as pd
    nodes = pd.read_parquet(os.path.join(PROCESSED_DIR, "vessel_node_features.parquet"))
    print(f"\n  Nodes: {len(nodes):,} vessels × {len(nodes.columns)} cols")
    print(f"  Labels: {nodes['vessel_iuu_label'].value_counts().to_dict()}")

if all_ok:
    print("\n🎉 ALL OUTPUT VERIFIED — READY FOR MODEL TRAINING")
else:
    print("\n⚠️  Some files missing — check pipeline errors above")

In [ ]:
# Cell 7: GPU check for model training
try:
    import subprocess
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print("GPU available:")
        print(result.stdout[:500])
    else:
        print("No GPU detected — CPU training will work but slower")
except:
    print("No GPU detected — CPU training will work but slower")

# Check Python version
import sys
print(f"Python: {sys.version}")
print(f"Working dir: {REPO_DIR}")

In [ ]:
# Cell 8: Train ST-GAT model
# Default: 100 epochs with early stopping (patience=15)
# Adjust --epochs for faster iteration during development
TRAIN_EPOCHS = 100

print(f"Training ST-GAT for up to {TRAIN_EPOCHS} epochs...")
print("This will take 20-60 minutes on CPU.\n")

result = subprocess.run(
    [sys.executable, os.path.join(REPO_DIR, "scripts", "train.py"),
     "--epochs", str(TRAIN_EPOCHS)],
    cwd=REPO_DIR,
    capture_output=False,
    timeout=7200  # 2 hours max
)

if result.returncode == 0:
    print("\n✅ Training complete!")
else:
    print(f"\n❌ Training failed with exit code {result.returncode}")

## Phase 8: Model Training (Optional)

Run the ST-GAT training. This takes 20-60 minutes on CPU depending on epochs.